# 00 · Simulate the dataset

**Purpose.** Generate the synthetic data used throughout this project: a *city-level panel* with a non-random, staggered rollout of a geo-targeted ad campaign (plus spillover between neighbouring cities), and a *user-level experiment* that is properly randomised inside the treated cities.

Because we write the data-generating process ourselves, we know the true effects. That is what lets later notebooks check whether each analysis method recovers the truth, rather than just producing a plausible-looking number.

**Outputs** (written to `data/raw/`)

| File | One row per | Rows |
|---|---|---|
| `city_panel.csv` | city × month | 4,500 |
| `rct_users.csv` | user | 150,000 |

**Running it.** *Restart kernel → Run All.* It takes well under a minute. Everything random is seeded, so every run produces identical data. The city-coordinates file is downloaded automatically if it is not already in `data/raw/`.

**Convention.** Columns prefixed `gt_` are *ground truth*: internals of the data generator that a real analyst would never see. Analysis notebooks must not use them as inputs. They exist only to check results against the truth at the end.

**Contents**
1. Setup and parameters
2. Cities
3. The confounder and the rollout schedule
4. City × month panel
5. Baseline outcome and seasonality
6. Spillover between neighbouring cities
7. Assembling the outcome
8. Nested randomised experiment (user level)
9. Ground-truth benchmarks
10. Save outputs and data dictionary

## 1. Setup and parameters

Every modelling choice lives in the parameter cell below, with the reasoning beside it, so the rest of the notebook contains no unexplained "magic numbers". To explore a different scenario, change a value here and re-run everything.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

In [2]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42  # re-set before each random step, so every step is reproducible on its own

# ── Cities ────────────────────────────────────────────────────────────────────
N_CITIES = 300
# ISO country codes counted as "Europe" and "North America" (the platform's core markets).
# Judgement calls: Russia is excluded (mostly Asia); North America = US, CA, MX only.
EUROPE = ["AL", "AD", "AT", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR",
          "DE", "GR", "HU", "IS", "IE", "IT", "LV", "LI", "LT", "LU", "MT", "MD", "MC", "ME",
          "NL", "MK", "NO", "PL", "PT", "RO", "RS", "SK", "SI", "ES", "SE", "CH", "UA", "GB"]
NORTH_AMERICA = ["US", "CA", "MX"]

# ── Timeline: 15 monthly periods ──────────────────────────────────────────────
START_MONTH = "2025-01-01"
END_MONTH = "2026-03-01"
ROLLOUT_START = "2025-07-01"  # first go-live; leaves 6 clean pre-period months (needed later for CUPED)
N_WAVES = 6                   # one wave per month, Jul-Dec 2025, so every city has at least 3 months post-launch

# ── Confounder: growth potential ──────────────────────────────────────────────
GP_MEAN, GP_SD = 50, 15       # normal(50, 15): most cities near average, a few stand out
GROWTH_SLOPE = 0.002          # retention gained per growth-potential point above average (+3pp at 1 SD).
                              # Sized to be comparable to TRUE_EFFECT, so the resulting bias is material.

# ── City-level outcome (retention rate) ───────────────────────────────────────
BASELINE_RETENTION = 0.25     # an average city: no campaign, no seasonality
TRUE_EFFECT = 0.04            # direct campaign effect: +4 percentage points
SEASONAL_EFFECT = {           # shared by every city; follows the academic calendar
    1: 0.02,  2: 0.00,  3: 0.00,  4: 0.00,  5: -0.01,  6: -0.03,   # Jan resurgence ... summer lull begins
    7: -0.04, 8: -0.03, 9: 0.03, 10: 0.01, 11: 0.00, 12: -0.01,    # summer lull ... back-to-school bump
}

# ── Spillover: ad reach leaking into neighbouring cities ──────────────────────
DECAY_DISTANCE_KM = 100       # weight = exp(-distance / 100): ~0.37 at 100 km, ~0.05 at 300 km
SPILLOVER_CAP = 0.015         # ceiling on the bump: 1.5pp, deliberately well below TRUE_EFFECT
SPILLOVER_SCALE = 2           # how fast exposure saturates; calibrated to the range of simulated exposure

# ── Nested randomised experiment ──────────────────────────────────────────────
USERS_PER_CITY = 500          # a practical default, not power-calculated (Module 5 checks whether it suffices)
NESTED_EFFECT = 0.02          # enhanced variant B: +2pp, an incremental change on top of the campaign

# ── Paths (this notebook lives in notebooks/) ─────────────────────────────────
DATA_RAW = Path("../data/raw")

## 2. Cities

City coordinates come from a GeoNames-derived list of cities with population of at least 15,000 ([joelacus/world-cities](https://github.com/joelacus/world-cities), CC BY 4.0). We keep Europe and North America, then draw a random sample of `N_CITIES`.

Each city gets a `city_id`. **Every join below uses `city_id`, never the city name.** Names are not unique (there are many Lexingtons), and a join on a non-unique key silently duplicates rows.

In [3]:
CITIES_URL = "https://raw.githubusercontent.com/joelacus/world-cities/main/world_cities_15000.csv"
CITIES_PATH = DATA_RAW / "world_cities.csv"

DATA_RAW.mkdir(parents=True, exist_ok=True)
if not CITIES_PATH.exists():          # download once; later runs reuse the local copy
    urlretrieve(CITIES_URL, CITIES_PATH)

world_cities = pd.read_csv(CITIES_PATH)
candidates = world_cities[world_cities["country"].isin(EUROPE + NORTH_AMERICA)]
print(f"{len(world_cities):,} cities worldwide -> {len(candidates):,} in Europe / North America")

34,135 cities worldwide -> 11,588 in Europe / North America


In [4]:
cities = candidates.sample(n=N_CITIES, random_state=SEED).reset_index(drop=True)
cities.insert(0, "city_id", cities.index)   # 0..299, a stable unique key

assert len(cities) == N_CITIES
assert cities["city_id"].is_unique
cities.head()

,city_id,country,name,lat,lng
0,0,NO,Grimstad,58.34050,8.59343
1,1,US,Lexington,42.44732,-71.22450
2,2,DE,Weißensee,52.55632,13.46649
3,3,IE,Dún Laoghaire,53.29395,-6.13586
4,4,US,Huntington,40.88310,-85.49748


## 3. The confounder and the rollout schedule

`growth_potential` stands in for the business reasons that decide which markets get the campaign first (market size, momentum). It does two jobs, and that overlap is the whole point:

1. it determines **rollout order**: higher potential, earlier launch;
2. it also **raises baseline retention** (section 5), independently of any campaign.

A variable that drives both treatment and outcome is a *confounder*. It makes treated cities look better even if the campaign did nothing.

Cities are split into `N_WAVES` equal groups by growth potential. Wave 0 (highest potential) launches first.

In [5]:
np.random.seed(SEED)
cities["growth_potential"] = np.random.normal(GP_MEAN, GP_SD, size=N_CITIES)

# qcut numbers the groups 0..5 from LOWEST to highest potential; flip so wave 0 = highest = first to launch
cities["rollout_wave"] = (N_WAVES - 1) - pd.qcut(cities["growth_potential"], q=N_WAVES, labels=False)

go_live_by_wave = dict(enumerate(pd.date_range(ROLLOUT_START, periods=N_WAVES, freq="MS")))
cities["go_live_date"] = cities["rollout_wave"].map(go_live_by_wave)

In [6]:
# Check: equal-sized waves, and average growth potential falls as launch gets later
assert (cities["rollout_wave"].value_counts() == N_CITIES // N_WAVES).all()

(cities.groupby("rollout_wave")
       .agg(n_cities=("city_id", "size"),
            mean_growth_potential=("growth_potential", "mean"),
            go_live=("go_live_date", "first"))
       .round({"mean_growth_potential": 1}))

,n_cities,mean_growth_potential,go_live
rollout_wave,,,
0,50,72.0,2025-07-01
1,50,59.3,2025-08-01
2,50,53.3,2025-09-01
3,50,46.8,2025-10-01
4,50,39.7,2025-11-01
5,50,28.3,2025-12-01


## 4. City × month panel

Cross-join the cities with the 15 months to get one row per city per month, then flag `treated` from the month a city reaches its go-live date.

Note the order: **all city-level attributes are created before this join**, so nothing has to be re-merged afterwards. (A column added to `cities` after the panel is built does not reach the panel.)

In [7]:
months = pd.date_range(START_MONTH, END_MONTH, freq="MS")
panel = cities.merge(pd.DataFrame({"month": months}), how="cross")
panel["treated"] = panel["month"] >= panel["go_live_date"]

# Grain check: exactly one row per city-month
assert len(panel) == N_CITIES * len(months)
assert not panel.duplicated(["city_id", "month"]).any()
print(f"{len(panel):,} rows = {N_CITIES} cities x {len(months)} months")

4,500 rows = 300 cities x 15 months


In [8]:
# Check: share of cities live each month. Should be 0 until July, then rise by one-sixth a month to 100%.
panel.groupby("month")["treated"].mean().round(2)

month
2025-01-01    0.00
2025-02-01    0.00
2025-03-01    0.00
2025-04-01    0.00
2025-05-01    0.00
2025-06-01    0.00
2025-07-01    0.17
2025-08-01    0.33
2025-09-01    0.50
2025-10-01    0.67
2025-11-01    0.83
2025-12-01    1.00
2026-01-01    1.00
2026-02-01    1.00
2026-03-01    1.00
Name: treated, dtype: float64

## 5. Baseline outcome and seasonality

Two ground-truth components of retention that exist regardless of the campaign:

- **Baseline.** `BASELINE_RETENTION`, shifted up or down by growth potential relative to the average city (`GP_MEAN`). A city with exactly average potential gets exactly the baseline. *This is the confounding.*
- **Seasonality.** A calendar-month effect shared by every city. Rollout timing is deliberately independent of season, so growth potential remains the only confounder.

In [9]:
panel["gt_baseline_outcome"] = BASELINE_RETENTION + (panel["growth_potential"] - GP_MEAN) * GROWTH_SLOPE
panel["gt_seasonal_effect"] = panel["month"].dt.month.map(SEASONAL_EFFECT)

assert panel["gt_seasonal_effect"].notna().all()   # every calendar month has an entry

## 6. Spillover between neighbouring cities

Ad targeting is imperfect, so campaign reach leaks into nearby cities. For each city and month:

1. compute the distance between every pair of cities (haversine formula, real coordinates);
2. turn distance into a weight that decays smoothly with distance;
3. sum the weights of all *currently treated* neighbours, giving a raw exposure score;
4. pass that through a saturating curve, so exposure has diminishing returns up to a ceiling (`SPILLOVER_CAP`);
5. apply the result **only to cities that are not yet treated themselves**. Spillover is meant to represent contamination of would-be control cities.

### 6.1 Distances

A function is only trustworthy once it has been tested against a known answer, so the haversine function is checked against the London to Paris distance (about 344 km) before it is used.

In [10]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km between points given in degrees. Works on single values or whole columns."""
    earth_radius_km = 6371
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * earth_radius_km * np.arcsin(np.sqrt(a))


# Test against a known answer: London (51.5074, -0.1278) to Paris (48.8566, 2.3522)
assert abs(haversine_km(51.5074, -0.1278, 48.8566, 2.3522) - 344) < 3

In [11]:
coords = cities[["city_id", "lat", "lng"]]
city_pairs = coords.merge(coords, how="cross", suffixes=("_a", "_b"))
city_pairs = city_pairs[city_pairs["city_id_a"] != city_pairs["city_id_b"]].copy()   # drop a city paired with itself

city_pairs["distance_km"] = haversine_km(city_pairs["lat_a"], city_pairs["lng_a"],
                                         city_pairs["lat_b"], city_pairs["lng_b"])
city_pairs["decay_weight"] = np.exp(-city_pairs["distance_km"] / DECAY_DISTANCE_KM)
city_pairs = city_pairs[["city_id_a", "city_id_b", "distance_km", "decay_weight"]]

assert len(city_pairs) == N_CITIES * (N_CITIES - 1)
city_pairs.sort_values("distance_km").head()

,city_id_a,city_id_b,distance_km,decay_weight
23834,79,134,0.602786,0.993990
40279,134,79,0.602786,0.993990
49174,163,274,0.905618,0.990985
82363,274,163,0.905618,0.990985
63659,212,59,1.359794,0.986494


### 6.2 Exposure from treated neighbours

Distances never change, but treatment status does, so each pair is repeated for every month together with city A's treatment status in that month. A treated neighbour contributes its decay weight, an untreated one contributes zero. Summing over all of city B's neighbours gives B's raw exposure each month.

In [12]:
treated_status = panel[["city_id", "month", "treated"]].rename(
    columns={"city_id": "city_id_a", "treated": "treated_a"})

exposure = city_pairs.merge(treated_status, on="city_id_a")        # one row per (pair, month)
exposure["weighted_signal"] = exposure["decay_weight"] * exposure["treated_a"]

raw_exposure = (exposure.groupby(["city_id_b", "month"])["weighted_signal"].sum()
                        .rename("raw_spillover_score")
                        .reset_index()
                        .rename(columns={"city_id_b": "city_id"}))

panel = panel.merge(raw_exposure, on=["city_id", "month"], how="left")
assert panel["raw_spillover_score"].notna().all()
panel["raw_spillover_score"].describe().round(2)

count    4500.00
mean        1.21
std         1.80
min         0.00
25%         0.00
50%         0.22
75%         1.84
max         8.30
Name: raw_spillover_score, dtype: float64

### 6.3 Saturating transform

`cap × (1 − exp(−score / scale))` rises from 0 towards `SPILLOVER_CAP` but never exceeds it, so a city surrounded by many treated neighbours gets diminishing returns rather than an ever-growing bump. This is preferred to a hard cut-off, which would need an arbitrary threshold.

Two columns result: the bump a city **would** receive from its neighbours (`gt_spillover_potential`), and the bump it **actually** receives (`gt_spillover_bump`), which is zero once the city is treated itself. Keeping both makes the true effect of treatment computable in section 9.

In [13]:
panel["gt_spillover_potential"] = SPILLOVER_CAP * (1 - np.exp(-panel["raw_spillover_score"] / SPILLOVER_SCALE))
panel["gt_spillover_bump"] = panel["gt_spillover_potential"] * (~panel["treated"])   # untreated cities only

assert (panel.loc[panel["treated"], "gt_spillover_bump"] == 0).all()
assert panel["gt_spillover_bump"].between(0, SPILLOVER_CAP).all()

## 7. Assembling the outcome

The city-month retention rate is the sum of its components:

`outcome = baseline + seasonal effect + TRUE_EFFECT × treated + spillover bump`

Assembling it in one place keeps the data-generating process readable. From here on, `outcome` and `treated` are "the data", and the `gt_` columns are off limits to analysis.

In [14]:
panel["outcome"] = (panel["gt_baseline_outcome"]
                    + panel["gt_seasonal_effect"]
                    + TRUE_EFFECT * panel["treated"]
                    + panel["gt_spillover_bump"])

assert panel["outcome"].between(0, 1).all()    # it is a rate
assert panel.isna().sum().sum() == 0           # no missing values anywhere
panel["outcome"].describe().round(3)

count    4500.000
mean        0.265
std         0.044
min         0.113
25%         0.232
50%         0.266
75%         0.297
max         0.436
Name: outcome, dtype: float64

## 8. Nested randomised experiment (user level)

Inside the treated cities, a second, cleanly randomised layer: users are assigned 50/50 to two versions of a follow-up onboarding experience (standard **A** vs enhanced **B**). Because assignment is random, the two groups differ only in the variant they saw. This is the clean comparison that contrasts with the confounded city-level one.

This table is deliberately separate from the panel. It is a different unit of analysis (a person, not a city-month), and mixing the two in one table would misrepresent the data's structure.

Each user's probability of being retained is their city's baseline (same formula as the panel, so cities vary realistically) plus `NESTED_EFFECT` if in variant B. The observed outcome is then a single weighted coin flip per user (a Bernoulli trial), which is where real sampling noise enters.

In [15]:
user_numbers = pd.DataFrame({"user_num": range(1, USERS_PER_CITY + 1)})
rct_users = cities[["city_id", "name", "growth_potential"]].merge(user_numbers, how="cross")
rct_users.insert(0, "user_id", range(len(rct_users)))      # unique key; (city_id, user_num) is the natural one

np.random.seed(SEED)
rct_users["variant"] = np.random.choice(["A", "B"], size=len(rct_users))

rct_users["gt_baseline_prob"] = BASELINE_RETENTION + (rct_users["growth_potential"] - GP_MEAN) * GROWTH_SLOPE
rct_users["gt_true_prob"] = rct_users["gt_baseline_prob"] + NESTED_EFFECT * (rct_users["variant"] == "B")

np.random.seed(SEED)
rct_users["retained"] = np.random.binomial(1, rct_users["gt_true_prob"])

assert len(rct_users) == N_CITIES * USERS_PER_CITY
assert rct_users["gt_true_prob"].between(0, 1).all()
assert rct_users["variant"].value_counts(normalize=True).between(0.49, 0.51).all()   # roughly 50/50
rct_users.head()

,user_id,city_id,name,growth_potential,user_num,variant,gt_baseline_prob,gt_true_prob,retained
0,0,0,Grimstad,57.450712,1,A,0.264901,0.264901,0
1,1,0,Grimstad,57.450712,2,B,0.264901,0.284901,1
2,2,0,Grimstad,57.450712,3,A,0.264901,0.264901,0
3,3,0,Grimstad,57.450712,4,A,0.264901,0.264901,0
4,4,0,Grimstad,57.450712,5,A,0.264901,0.264901,0


## 9. Ground-truth benchmarks

What a correct method *should* recover in later modules.

- **Nested experiment.** `NESTED_EFFECT` (+2pp) in expectation. The realised gap differs a little through sampling noise.
- **City-level campaign.** The parameter is +4pp, but the *effective* effect of treating a city is slightly smaller. Spillover is switched off for treated cities, so a city that gets treated also gives up the spillover it would otherwise have received. The true effect on a treated city-month is therefore `TRUE_EFFECT − gt_spillover_potential`.

Use the benchmark below, not the raw 4pp, when judging the naive and corrected estimates in Modules 3 and 4.

In [16]:
treated_rows = panel["treated"]
true_effect_on_treated = (TRUE_EFFECT - panel.loc[treated_rows, "gt_spillover_potential"]).mean()
realised_rct = rct_users.groupby("variant")["retained"].mean()

print(f"City campaign: parameter {TRUE_EFFECT:.1%}, true average effect on treated city-months {true_effect_on_treated:.2%}")
print(f"Nested test:   parameter {NESTED_EFFECT:.1%}, realised gap B - A {realised_rct['B'] - realised_rct['A']:.2%}")

City campaign: parameter 4.0%, true average effect on treated city-months 3.18%
Nested test:   parameter 2.0%, realised gap B - A 2.12%


## 10. Save outputs and data dictionary

Columns are written in an explicit order, observable ones first and `gt_` ones last. The assertion makes sure no column is silently left out or added.

In [17]:
panel_observable = ["city_id", "name", "country", "lat", "lng", "growth_potential", "rollout_wave",
                    "go_live_date", "month", "treated", "raw_spillover_score", "outcome"]
panel_truth = [c for c in panel.columns if c.startswith("gt_")]
assert set(panel.columns) == set(panel_observable + panel_truth)

rct_observable = ["user_id", "city_id", "name", "growth_potential", "user_num", "variant", "retained"]
rct_truth = [c for c in rct_users.columns if c.startswith("gt_")]
assert set(rct_users.columns) == set(rct_observable + rct_truth)

panel[panel_observable + panel_truth].to_csv(DATA_RAW / "city_panel.csv", index=False)
rct_users[rct_observable + rct_truth].to_csv(DATA_RAW / "rct_users.csv", index=False)

In [18]:
# Reload and verify what was written
check_panel = pd.read_csv(DATA_RAW / "city_panel.csv", parse_dates=["month", "go_live_date"])
check_rct = pd.read_csv(DATA_RAW / "rct_users.csv")
print(f"city_panel.csv: {check_panel.shape[0]:,} rows x {check_panel.shape[1]} columns")
print(f"rct_users.csv:  {check_rct.shape[0]:,} rows x {check_rct.shape[1]} columns")

city_panel.csv: 4,500 rows x 16 columns
rct_users.csv:  150,000 rows x 9 columns


### Data dictionary

**`city_panel.csv`** (one row per city × month)

| Column | Meaning | Visible to an analyst? |
|---|---|---|
| `city_id`, `name`, `country`, `lat`, `lng` | City identifier and location | Yes |
| `growth_potential` | City-level score that drives both rollout order and baseline retention (the confounder) | Yes |
| `rollout_wave`, `go_live_date` | Wave number (0 = first) and the month the campaign went live in the city | Yes |
| `month` | Calendar month (first day) | Yes |
| `treated` | Whether the campaign is live in the city that month | Yes |
| `raw_spillover_score` | Distance-weighted count of treated neighbours; derivable from coordinates and `treated` | Yes (derived) |
| `outcome` | Retention rate for the city-month | Yes |
| `gt_baseline_outcome` | Baseline retention before seasonality or campaign | **No** |
| `gt_seasonal_effect` | Calendar-month effect | **No** |
| `gt_spillover_potential` | Spillover a city would receive from its neighbours if untreated | **No** |
| `gt_spillover_bump` | Spillover actually received (zero once treated) | **No** |

**`rct_users.csv`** (one row per user)

| Column | Meaning | Visible to an analyst? |
|---|---|---|
| `user_id`, `user_num`, `city_id`, `name` | User and city identifiers | Yes |
| `growth_potential` | City-level score, carried to each user | Yes |
| `variant` | Randomly assigned version, A or B | Yes |
| `retained` | 1 if the user was retained, 0 otherwise | Yes |
| `gt_baseline_prob` | User's retention probability before the variant effect | **No** |
| `gt_true_prob` | User's retention probability including the variant effect | **No** |

**Next:** `01_eda.ipynb`, which loads these two files and checks that the bias and spillover built in here are visible in the data.